# 215 — Generate Embeddings

Generates embeddings for all `:Chunk` nodes via OpenAI `text-embedding-3-small` (1 536 dimensions),
writes them to Neo4j, and creates `SEMANTICALLY_SIMILAR` edges between semantically similar chunks
from **different** documents (cosine similarity > 0.85).

Re-runnable: yes — skips chunks that already have embeddings.

In [1]:
import sys, os, time
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

project_root = Path(os.getcwd()).parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

load_dotenv(project_root / '.env')

from src.graph.connection import Neo4jConnection

client_oai    = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
EMBED_MODEL   = 'text-embedding-3-small'
DIMENSIONS    = 1536
BATCH_SIZE    = 50
SIM_THRESHOLD = 0.85
SIM_TOP_K     = 10   # neighbours to query per chunk

conn = Neo4jConnection()
conn.connect()
print('Connected to Neo4j')

Connected to Neo4j


## Step 1 — Create vector index

In [2]:
conn.run_query(
    'CREATE VECTOR INDEX chunk_embeddings IF NOT EXISTS '
    'FOR (c:Chunk) ON c.embedding '
    'OPTIONS {indexConfig: {`vector.dimensions`: $dims, `vector.similarity_function`: $fn}}',
    {'dims': DIMENSIONS, 'fn': 'cosine'},
)
print('Vector index chunk_embeddings created (or already exists).')

# Wait for index to come online
import time as _time
for _ in range(10):
    result = conn.run_query(
        "SHOW INDEXES YIELD name, state WHERE name = 'chunk_embeddings'",
    )
    if result and result[0].get('state') == 'ONLINE':
        print('Index is ONLINE.')
        break
    print('Waiting for index to come online...')
    _time.sleep(2)
else:
    print('\u26a0  Index may not be online yet. Proceed with caution.')


Vector index chunk_embeddings created (or already exists).
Index is ONLINE.


## Step 2 — Fetch chunks without embeddings

In [3]:
chunks = conn.run_query(
    'MATCH (c:Chunk) WHERE c.embedding IS NULL AND c.text IS NOT NULL '
    'RETURN c.chunk_id AS chunk_id, c.text AS text, c.source_document AS source_document'
)
print(f'Chunks without embeddings: {len(chunks)}')
if chunks:
    print(f'  Sample: {chunks[0]["chunk_id"]}')

Chunks without embeddings: 189
  Sample: APS-112-CHUNK-0043


## Step 3 — Generate and write embeddings

In [4]:
def get_embeddings(texts: list) -> list:
    '''Call OpenAI embeddings API for a batch of texts.'''
    response = client_oai.embeddings.create(
        input=texts,
        model=EMBED_MODEL,
    )
    return [item.embedding for item in response.data]


total_written = 0

for i in range(0, len(chunks), BATCH_SIZE):
    batch      = chunks[i:i + BATCH_SIZE]
    texts      = [c['text'] for c in batch]
    chunk_ids  = [c['chunk_id'] for c in batch]

    print(f'Batch {i // BATCH_SIZE + 1}: embedding {len(batch)} chunks...', end=' ')
    try:
        embeddings = get_embeddings(texts)
    except Exception as e:
        print(f'\u2717 Error: {e}')
        continue

    rows = [{'chunk_id': cid, 'embedding': emb} for cid, emb in zip(chunk_ids, embeddings)]
    conn.run_query(
        'UNWIND $rows AS row '
        'MATCH (c:Chunk {chunk_id: row.chunk_id}) '
        'SET c.embedding = row.embedding',
        {'rows': rows},
    )
    total_written += len(batch)
    print(f'\u2713')

    if i + BATCH_SIZE < len(chunks):
        time.sleep(0.5)  # gentle rate-limit buffer

print(f'\nEmbeddings written: {total_written}')

Batch 1: embedding 50 chunks... ✓
Batch 2: embedding 50 chunks... ✓
Batch 3: embedding 50 chunks... ✓
Batch 4: embedding 39 chunks... ✓

Embeddings written: 189


## Step 4 — Create SEMANTICALLY_SIMILAR edges

In [5]:
# For each chunk, find top-k neighbours from different source documents with cosine > SIM_THRESHOLD
# Uses the Neo4j vector index for efficient approximate nearest-neighbour search

print(f'Creating SEMANTICALLY_SIMILAR edges (cosine > {SIM_THRESHOLD}, cross-document only)...')

result = conn.run_query(
    '''
    MATCH (c:Chunk)
    WHERE c.embedding IS NOT NULL
    CALL db.index.vector.queryNodes('chunk_embeddings', $k, c.embedding)
      YIELD node AS neighbour, score
    WHERE score > $threshold
      AND neighbour.source_document <> c.source_document
      AND neighbour.chunk_id <> c.chunk_id
    MERGE (c)-[r:SEMANTICALLY_SIMILAR]->(neighbour)
    SET r.score = score
    RETURN count(r) AS edges_created
    ''',
    {'k': SIM_TOP_K, 'threshold': SIM_THRESHOLD},
)
sim_count = result[0]['edges_created'] if result else 0
print(f'SEMANTICALLY_SIMILAR edges created: {sim_count}')

Creating SEMANTICALLY_SIMILAR edges (cosine > 0.85, cross-document only)...
SEMANTICALLY_SIMILAR edges created: 126


## Step 5 — Summary

In [6]:
embedded_count = conn.run_query(
    'MATCH (c:Chunk) WHERE c.embedding IS NOT NULL RETURN count(c) AS n'
)[0]['n']
null_count = conn.run_query(
    'MATCH (c:Chunk) WHERE c.embedding IS NULL RETURN count(c) AS n'
)[0]['n']
sim_count = conn.run_query(
    'MATCH ()-[r:SEMANTICALLY_SIMILAR]->() RETURN count(r) AS n'
)[0]['n']

print(f'Chunks with embeddings    : {embedded_count}')
print(f'Chunks without embeddings : {null_count}')
print(f'SEMANTICALLY_SIMILAR edges          : {sim_count}')

conn.close()
print('\nConnection closed.')

Chunks with embeddings    : 189
Chunks without embeddings : 0
SEMANTICALLY_SIMILAR edges          : 126

Connection closed.
